# ਪਾਠ 11 - ਏਜੰਟ-ਟੂ-ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੋਲ


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ?

The **ਏਜੰਟ-ਟੁ-ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੋਲ** ਇੱਕ ਖੁੱਲ੍ਹਾ ਮਿਆਰੀਕਰਨ ਹੈ ਜੋ AI ਏਜੰਟਾਂ ਨੂੰ ਗੱਲਬਾਤ ਕਰਨ,
ਇੱਕ ਦੂਜੇ ਨੂੰ ਲੱਭਣ, ਅਤੇ ਸਹਿਯੋਗ ਕਰਨ — ਭਾਵੇਂ ਉਹ ਵੱਖ-ਵੱਖ ਫਰੇਮਵਰਕਾਂ 'ਤੇ ਬਣੇ ਹੋਣ ਜਾਂ
ਵੱਖ-ਵੱਖ ਸੇਵਾਵਾਂ ਵੱਲੋਂ ਹੋਸਟ ਕੀਤੇ ਗਏ ਹੋਣ।

Key concepts:

- **ਖੋਜ** – ਏਜੰਟ ਇੱਕ *ਏਜੰਟ ਕਾਰਡ* ਪ੍ਰਕਾਸ਼ਿਤ ਕਰਦੇ ਹਨ ਜੋ ਉਹਨਾਂ ਦੀਆਂ ਯੋਗਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰਦਾ ਹੈ, ਜਿਸ ਨਾਲ
  ਹੋਰ ਏਜੰਟਾਂ (ਜਾਂ ਸੰਚਾਲਕ) ਲਈ ਕਿਸੇ ਟਾਸਕ ਲਈ ਸਹੀ ਵਿਸ਼ੇਸ਼ਜ्ञ ਨੂੰ ਲੱਭਣਾ ਆਸਾਨ ਬਣਾਉਂਦਾ ਹੈ।
- **ਸੁਨੇਹਾ-ਸੰਚਾਰ** – ਏਜੰਟ ਇੱਕ ਆਮ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਸੰਰਚਿਤ ਸੁਨੇਹੇ ਅਦਲਾ-ਬਦਲੀ ਕਰਦੇ ਹਨ, ਤਾਂ ਜੋ
  ਇੱਕ ਏਜੰਟ ਦੀ ਬੇਨਤੀ ਦੂਜੇ ਵੱਲੋਂ ਸਮਝੀ ਅਤੇ ਪੂਰੀ ਕੀਤੀ ਜਾ ਸਕੇ ਭਾਵੇਂ ਉਸਦੀ ਅੰਦਰੂਨੀ
  ਕਾਰਗੁਜ਼ਾਰੀ ਜੋ ਵੀ ਹੋਵੇ।
- **ਕੰਮ ਦਾ ਜੀਵਨਚੱਕਰ** – A2A ਕੁਝ ਹਾਲਤਾਂ ਦੀ ਪਰਿਭਾਸ਼ਾ ਕਰਦਾ ਹੈ ਜਿਵੇਂ ਕਿ *ਜਮ੍ਹਾਂ ਕੀਤਾ*, *ਕੰਮ ਕਰ ਰਿਹਾ*, *ਪੂਰਾ*, ਅਤੇ
  *ਅਸਫਲ*, ਸੰਚਾਲਕ ਨੂੰ ਇਸ ਗੱਲ ਦੀ ਪੂਰੀ ਦਿੱਖ ਦਿੰਦਾ ਹੈ ਕਿ ਕਿਸ ਤਰ੍ਹਾਂ ਸੌਂਪਿਆ ਗਿਆ ਕੰਮ ਅੱਗੇ ਵੱਧ ਰਿਹਾ ਹੈ।

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## ਵਿਸ਼ੇਸ਼ ਯਾਤਰਾ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ਵਰਕਫਲੋ ਰਾਹੀਂ ਬਹੁ-ਏਜੰਟ ਸਹਿਯੋਗ

ਅਸੀਂ ਤਿੰਨ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜਦੇ ਹਾਂ ਜੋ A2A ਸੁਨੇਹਾ-ਪਾਸਿੰਗ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ:

1. **CurrencyExchangeAgent** ਯੂਜ਼ਰ ਦੀ ਬੇਨਤੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਮੁਦਰਾ ਸਬੰਧੀ ਮਾਰਗਦਰਸ਼ਨ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. **ActivityPlannerAgent** ਸੰਵਰਧਿਤ ਸੰਦਰਭ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਗਤੀਵਿਧੀ ਦੀਆਂ ਸਿਫਾਰਸ਼ਾਂ ਜੋੜਦਾ ਹੈ।
3. **TravelManagerAgent** ਦੋਵਾਂ ਇਨਪੁਟਾਂ ਨੂੰ ਇੱਕਠਾ ਕਰਕੇ ਅੰਤਿਮ ਯਾਤਰਾ ਸਾਰ ਤਿਆਰ ਕਰਦਾ ਹੈ।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਪ੍ਰੋਡਕਸ਼ਨ ਵਿੱਚ A2A ਦੀ ਸਮਝ

ਇੱਕ ਪ੍ਰੋਡਕਸ਼ਨ ਵਾਤਾਵਰਨ ਵਿੱਚ A2A ਪ੍ਰੋਟੋਕਾਲ ਸ਼ਕਤੀਸ਼ਾਲੀ ਕ੍ਰਾਸ-ਸੇਵਾ ਸੈਨਾਰਿਓਜ਼ ਨੂੰ ਖੋਲ੍ਹਦਾ ਹੈ:

| Capability | Description |
|---|---|
| **Cross-framework interop** | ਇੱਕ ਫਰੇਮਵਰਕ ਨਾਲ ਬਣਾਇਆ ਗਇਆ ਏਜੰਟ ਕਿਸੇ ਹੋਰ A2A-ਅਨੁਕੂਲ ਫਰੇਮਵਰਕ ਨਾਲ ਬਣੇ ਏਜੰਟ ਨੂੰ ਕੰਮ ਸੌਂਪ ਸਕਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਅਸਲ ਵਿੱਚ ਸੰਸਥਾਵਾਂ-ਪਾਰ ਇੰਟਰਓਪਰੇਬਿਲਿਟੀ ਸੰਭਵ ਹੁੰਦੀ ਹੈ। |
| **Service boundaries** | ਏਜੰਟ ਵੱਖ-ਵੱਖ ਮਾਈਕ੍ਰੋਸਰਵਿਸਿਜ਼, ਕਲਾਉਡ ਰੀਜੀਅਨ ਜਾਂ ਇੱਥੋਂ ਤੱਕ ਕਿ ਵੱਖ-ਵੱਖ ਸੰਗਠਨਾਂ ਵਿੱਚ ਵੱਸ ਸਕਦੇ ਹਨ ਅਤੇ ਫਿਰ ਵੀ ਬੇਰੁਕਾਵਟ ਤਰੀਕੇ ਨਾਲ ਸਹਿਯੋਗ ਕਰ ਸਕਦੇ ਹਨ। |
| **Dynamic discovery** | ਇੱਕ ਆਰਕੇਸਟਰੇਟਰ ਰਨਟਾਈਮ 'ਤੇ ਏਜੰਟ ਕਾਰਡ ਰਜਿਸਟਰੀ ਨੂੰ ਕਵੈਰੀ ਕਰ ਕੇ ਦਿੱਤੇ ਗਏ ਉਪ-ਕਾਰਜ ਲਈ ਸਭ ਤੋਂ ਉਚਿਤ ਮਾਹਿਰ ਲੱਭ ਸਕਦਾ ਹੈ। |
| **Streaming & push notifications** | A2A ਰੀਅਲ-ਟਾਈਮ ਪ੍ਰਗਤੀ ਅੱਪਡੇਟ ਲਈ Server-Sent Events (SSE) ਅਤੇ ਲੰਬੇ ਸਮੇਂ ਚੱਲਣ ਵਾਲੇ ਕਾਰਜਾਂ ਲਈ ਪੁਸ਼ ਸੂਚਨਾਵਾਂ ਦਾ ਸਮਰਥਨ ਕਰਦਾ ਹੈ। |

ਉਪਰ ਦਿੱਤਾ ਗਿਆ ਵਰਕਫਲੋਅ ਅਸੀਂ ਬਣਾਇਆ ਉਹ ਇਸ ਪੈਟਰਨ ਦਾ ਇੱਕ ਸਾਦਾ, ਇਨ-ਪ੍ਰੋਸੈਸ ਵਰਜ਼ਨ ਹੈ। ਅਸਲ ਡਿਪਲੋਇਮੈਂਟ ਵਿੱਚ ਹਰ ਏਜੰਟ ਇੱਕ HTTP ਐਂਡਪੁਆਇੰਟ ਖੋਲ੍ਹੇਗਾ, ਇੱਕ ਏਜੰਟ ਕਾਰਡ ਪ੍ਰਕਾਸ਼ਿਤ ਕਰੇਗਾ, ਅਤੇ A2A JSON-RPC ਪ੍ਰੋਟੋਕਾਲ ਰਾਹੀਂ ਸੰਚਾਰ ਕਰੇਗਾ।


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ:

1. **A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ** — ਏਜੰਟ-ਟੂ-ਏਜੰਟ ਖੋਜ, ਮੇਸੇਜਿੰਗ,
   ਅਤੇ ਟਾਸਕ ਪ੍ਰਬੰਧਨ.
2. **ਖਾਸ ਏਜੰਟ ਕਿਵੇਂ ਬਣਾਉਣੇ** — ਇੱਕ ਕਰੰਸੀ ਐਕਸਚੇਂਜ ਏਜੰਟ, ਇੱਕ ਐਕਟਿਵਿਟੀ ਪਲੈਨਰ ਏਜੰਟ, ਅਤੇ ਇੱਕ ਯਾਤਰਾ ਪ੍ਰਬੰਧਕ ਓਰਕੇਸਟਰੇਟਰ.
3. **ਏਜੰਟਾਂ ਨੂੰ ਵਰਕਫਲੋ ਵਿੱਚ ਕਿਵੇਂ ਜੋੜਨਾ** — `WorkflowBuilder` ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕ੍ਰਮਿਕ
   ਸੰਦੇਸ਼-ਪਾਸਿੰਗ ਨੂੰ ਏਜੰਟਾਂ ਦੇ ਵਿਚਕਾਰ ਮਾਡਲ ਕਰਨ ਲਈ.
4. **ਉਤਪਾਦਨ ਵਿੱਚ A2A ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ** — ਫਰੇਮਵਰਕ-ਪਾਰ, ਸੇਵਾ-ਪਾਰ ਸਹਿਯੋਗ ਨੂੰ
   ਡਾਇਨਾਮਿਕ ਖੋਜ ਅਤੇ ਸਟ੍ਰੀਮਿੰਗ ਅੱਪਡੇਟਸ ਨਾਲ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਸਪੱਸ਼ਟੀਕਰਨ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ [Co-op Translator](https://github.com/Azure/co-op-translator) ਨਾਮਕ ਏ.ਆਈ. ਅਨੁਵਾਦ ਸੇਵਾ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਪਰ ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸੁਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਮੌਜੂਦ ਮੂਲ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਅਧਿਕਾਰਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਂ ਨਿਰਣਾਇਕ ਜਾਣਕਾਰੀ ਲਈ ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਕਾਰਨ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
